# 00 — Setup: catalog, schema, gold tables

Creates the Unity Catalog assets the rest of the demo depends on.

**Tables produced:**
- `advertisers` — 9 advertisers, the tenant registry
- `campaign_daily_perf` — daily delivery + spend per campaign, 12 weeks
- `audience_segments` — segments per advertiser with size + overlap score
- `pacing_metrics` — flight-to-date pacing per campaign

Run via Databricks Connect from local **or** as a workspace notebook on serverless.

In [ ]:
import os
from datetime import date, timedelta
import random

import polars as pl

CATALOG = os.environ.get('CATALOG', 'alexander_booth')
SCHEMA  = os.environ.get('SCHEMA',  'maas_summit_team8')
SEED    = 20260520
random.seed(SEED)

print(f'Target: {CATALOG}.{SCHEMA}')

In [ ]:
# Connect to the workspace. Use serverless if running locally via Databricks Connect.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print('catalog and schema ready')

## Advertiser registry

9 advertisers across 3 tiers. `tier` drives Lakebase scale-to-zero behavior in the cost panel: enterprise = always warm, growth = warm during business hours, starter = scale to zero.

In [ ]:
advertisers = [
    # advertiser_id, name, tier, brand_color, monthly_budget_usd
    ('adv_patagonia', 'Patagonia',    'enterprise', '#1f5e3a', 850_000),
    ('adv_allbirds',  'Allbirds',     'enterprise', '#5fb3a1', 620_000),
    ('adv_bombas',    'Bombas',       'growth',     '#c3423f',  410_000),
    ('adv_warbyparker','Warby Parker','enterprise', '#0a3d62', 920_000),
    ('adv_glossier',  'Glossier',     'growth',     '#f8b4b4', 380_000),
    ('adv_outdoorvoices','Outdoor Voices','growth', '#f59e0b', 290_000),
    ('adv_caraway',   'Caraway',      'starter',    '#94a89a', 140_000),
    ('adv_brightland','Brightland',   'starter',    '#e9b44c', 95_000),
    ('adv_olipop',    'Olipop',       'starter',    '#5b3a29', 120_000),
]
adv_df = pl.DataFrame(
    advertisers,
    schema=['advertiser_id', 'name', 'tier', 'brand_color', 'monthly_budget_usd'],
    orient='row',
)
adv_df

## Campaigns

~6 campaigns per advertiser, mix of channels (paid search, social, CTV, display). 50 total.

In [ ]:
CHANNELS = ['paid_search', 'paid_social', 'ctv', 'display', 'video', 'native']
OBJECTIVES = ['awareness', 'consideration', 'conversion', 'retention']

rng = random.Random(SEED)
campaigns = []
for adv_id, adv_name, tier, _, monthly_budget in advertisers:
    n_campaigns = {'enterprise': 7, 'growth': 6, 'starter': 4}[tier]
    daily_budget_total = monthly_budget / 30
    # split budget across campaigns log-normal
    weights = [rng.expovariate(1.0) for _ in range(n_campaigns)]
    s = sum(weights)
    for i, w in enumerate(weights):
        ch = rng.choice(CHANNELS)
        obj = rng.choice(OBJECTIVES)
        campaigns.append((
            f'{adv_id}_camp_{i:02d}',
            adv_id,
            f'{adv_name} {ch.replace("_", " ").title()} {obj.title()}',
            ch,
            obj,
            round(daily_budget_total * (w / s), 2),
        ))

camp_df = pl.DataFrame(
    campaigns,
    schema=['campaign_id', 'advertiser_id', 'name', 'channel', 'objective', 'daily_budget_usd'],
    orient='row',
)
print(f'{len(campaigns)} campaigns')
camp_df.head(8)

## Daily performance (12 weeks)

Realistic patterns: weekday/weekend dip, slight upward trend, channel-specific CTR/CVR, log-normal noise.

In [ ]:
import math

END_DATE = date(2026, 5, 18)
DAYS = 84  # 12 weeks

# Channel benchmarks (CPM, CTR, CVR)
CHANNEL_PERF = {
    'paid_search': (24, 0.045, 0.062),
    'paid_social': (8,  0.012, 0.018),
    'ctv':         (32, 0.008, 0.011),
    'display':     (3,  0.004, 0.006),
    'video':       (14, 0.010, 0.014),
    'native':      (6,  0.006, 0.012),
}
AOV = 78  # avg order value

rng = random.Random(SEED + 1)
rows = []
for camp_id, adv_id, _, channel, _, daily_budget in campaigns:
    cpm, ctr, cvr = CHANNEL_PERF[channel]
    for d in range(DAYS):
        the_date = END_DATE - timedelta(days=DAYS - 1 - d)
        # weekday/weekend factor
        wd_factor = 0.78 if the_date.weekday() >= 5 else 1.0
        # slight upward trend
        trend = 1.0 + 0.002 * d
        # noise (log-normal-ish)
        noise_spend = math.exp(rng.gauss(0, 0.18))
        spend = daily_budget * wd_factor * trend * noise_spend
        spend = round(spend, 2)
        impressions = int(spend / cpm * 1000)
        # CTR/CVR with their own noise
        eff_ctr = ctr * math.exp(rng.gauss(0, 0.12))
        eff_cvr = cvr * math.exp(rng.gauss(0, 0.15))
        clicks  = int(impressions * eff_ctr)
        conversions = int(clicks * eff_cvr)
        revenue = round(conversions * AOV * math.exp(rng.gauss(0, 0.08)), 2)
        rows.append((
            adv_id, camp_id, the_date, channel,
            impressions, clicks, conversions, spend, revenue,
        ))

perf_df = pl.DataFrame(
    rows,
    schema=['advertiser_id','campaign_id','date','channel',
            'impressions','clicks','conversions','spend_usd','revenue_usd'],
    orient='row',
)
print(f'{perf_df.height:,} daily rows')
perf_df.head(5)

## Audience segments

Per advertiser: ~4 named segments with size + overlap score vs. their other segments.

In [ ]:
SEGMENT_NAMES = [
    'High-Value Repeat Buyers', 'Cart Abandoners 30d', 'Lookalike Top 5%',
    'Loyalty Members', 'Brand Affinity Outdoor', 'Email Engaged 90d',
    'Site Visitors 14d', 'Wishlist Activated', 'Promo-Sensitive',
    'New-to-Brand Prospects',
]
rng = random.Random(SEED + 2)
segs = []
for adv_id, adv_name, tier, _, _ in advertisers:
    n = {'enterprise': 5, 'growth': 4, 'starter': 3}[tier]
    chosen = rng.sample(SEGMENT_NAMES, n)
    for i, name in enumerate(chosen):
        size = int(math.exp(rng.gauss(11, 0.9)))  # 5k-200k typical
        overlap = round(rng.betavariate(2, 5), 3)
        segs.append((adv_id, f'{adv_id}_seg_{i:02d}', name, size, overlap))

seg_df = pl.DataFrame(
    segs,
    schema=['advertiser_id','segment_id','name','size','overlap_score'],
    orient='row',
)
print(f'{seg_df.height} segments')
seg_df.head(6)

## Pacing metrics

Per campaign, flight-to-date spend vs budget — what every media planner stares at every morning.

In [ ]:
FLIGHT_DAYS = 30  # current month
today = END_DATE
rng = random.Random(SEED + 3)
pacing = []
for camp_id, adv_id, _, _, _, daily_budget in campaigns:
    monthly_budget = round(daily_budget * FLIGHT_DAYS, 2)
    days_into_flight = rng.randint(15, 28)
    days_remaining   = FLIGHT_DAYS - days_into_flight
    expected_pace = days_into_flight / FLIGHT_DAYS
    actual_pace   = expected_pace * math.exp(rng.gauss(0, 0.16))
    actual_pace   = max(0.05, min(actual_pace, 1.2))
    ftd_spend = round(monthly_budget * actual_pace, 2)
    health    = 'on_pace' if abs(actual_pace - expected_pace) < 0.07 else (
                'underpacing' if actual_pace < expected_pace else 'overpacing')
    pacing.append((adv_id, camp_id, today, monthly_budget, ftd_spend,
                   round(expected_pace, 3), round(actual_pace, 3),
                   days_remaining, health))

pace_df = pl.DataFrame(
    pacing,
    schema=['advertiser_id','campaign_id','as_of_date','monthly_budget_usd','ftd_spend_usd',
            'expected_pace','actual_pace','days_remaining','health'],
    orient='row',
)
pace_df.head(8)

## Write to Unity Catalog

Databricks Connect doesn't support MERGE — we use `INSERT OVERWRITE` via Spark DataFrames.

In [ ]:
def write_table(pl_df, name):
    sdf = spark.createDataFrame(pl_df.to_pandas())
    fq = f'{CATALOG}.{SCHEMA}.{name}'
    sdf.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(fq)
    n = spark.sql(f'SELECT COUNT(*) AS n FROM {fq}').collect()[0]['n']
    print(f'  {fq}: {n:,} rows')

print('writing gold tables...')
write_table(adv_df,  'advertisers')
write_table(camp_df, 'campaigns')
write_table(perf_df, 'campaign_daily_perf')
write_table(seg_df,  'audience_segments')
write_table(pace_df, 'pacing_metrics')
print('done')

## Sanity checks

Spot-check totals so the dashboards look right.

In [ ]:
spark.sql(f'''
  SELECT a.name AS advertiser,
         ROUND(SUM(p.spend_usd), 0) AS spend_12w,
         ROUND(SUM(p.revenue_usd), 0) AS revenue_12w,
         ROUND(SUM(p.revenue_usd) / NULLIF(SUM(p.spend_usd),0), 2) AS roas
  FROM {CATALOG}.{SCHEMA}.campaign_daily_perf p
  JOIN {CATALOG}.{SCHEMA}.advertisers a USING (advertiser_id)
  GROUP BY a.name
  ORDER BY spend_12w DESC
''').display() if hasattr(spark.sql(f'SELECT 1'), 'display') else spark.sql(f'''
  SELECT a.name AS advertiser,
         ROUND(SUM(p.spend_usd), 0) AS spend_12w,
         ROUND(SUM(p.revenue_usd), 0) AS revenue_12w,
         ROUND(SUM(p.revenue_usd) / NULLIF(SUM(p.spend_usd),0), 2) AS roas
  FROM {CATALOG}.{SCHEMA}.campaign_daily_perf p
  JOIN {CATALOG}.{SCHEMA}.advertisers a USING (advertiser_id)
  GROUP BY a.name
  ORDER BY spend_12w DESC
''').show()